# **MULAI EKSPERIMEN**

### **IMPORT LIBRARY**

In [ ]:
import dlib  # Library untuk deteksi wajah dan pengenalan fitur wajah
import cv2  # Library OpenCV untuk pemrosesan gambar
import numpy as np  # Library untuk operasi numerik
import os  # Library untuk manipulasi file dan direktori
from tqdm import tqdm  # Library untuk menampilkan progress bar
from sklearn.svm import SVC  # Model Support Vector Classifier dari scikit-learn
from sklearn.metrics import classification_report  # Untuk mengevaluasi hasil klasifikasi
import pickle  # Library untuk menyimpan dan memuat objek Python
from sklearn.model_selection import learning_curve
import matplotlib.pyplot as plt

### **DETEKSI DAN EKSTRAKSI WAJAH**

In [ ]:
##
# shape_predictor_68_face_landmarks.dat digunakan untuk deteksi landmark wajah (titik-titik penting seperti mata, hidung, mulut, dll.) setelah bounding box wajah ditemukan.
# dlib_face_recognition_resnet_model_v1.dat digunakan untuk ekstraksi fitur wajah dalam bentuk vektor 128 dimensi, yang kemudian digunakan untuk pengenalan wajah.

# Keduanya memiliki peran yang berbeda tetapi saling melengkapi:
# Bounding box wajah terlebih dahulu (hasil deteksi wajah dlib).
# Kemudian, bounding box ini digunakan oleh shape_predictor untuk mendeteksi landmark.
# Landmark ini memungkinkan wajah dilokalisasi lebih baik untuk diekstraksi oleh model ResNet (dlib_face_recognition_resnet_model_v1.dat).
# ##
class FaceFeatureExtractor:
    def __init__(self, predictor_path='./predictor/shape_predictor_68_face_landmarks.dat',
                 face_rec_model_path='./predictor/dlib_face_recognition_resnet_model_v1.dat'):
        # #
        # Inisialisasi FaceFeatureExtractor dengan model deteksi wajah, prediktor landmarks,
        # dan model pengenalan wajah dlib.
        # #

        # Detektor wajah dlib
        self.detector = dlib.get_frontal_face_detector()
        # Prediktor landmark untuk wajah
        self.shape_predictor = dlib.shape_predictor(predictor_path)
        # Model pengenalan wajah dlib (menghasilkan deskriptor wajah 128-D)
        self.face_rec_model = dlib.face_recognition_model_v1(face_rec_model_path)

    def extract_features(self, image_path):

        ##
        # Mengekstrak fitur wajah dan landmarks dari gambar yang diberikan.
        # Args:
        #    image_path (str): Path ke file gambar.
        # Returns:
        #    face_descriptor (np.array): Deskriptor wajah 128-D.
        #    landmarks (np.array): Koordinat landmarks wajah.
        #    success (bool): Indikator keberhasilan ekstraksi fitur.
        ##

        try:
            # Membaca gambar dari path
            img = cv2.imread(image_path)
            if img is None:
                return None, None, False

            # Konversi gambar dari BGR (OpenCV) ke RGB (dlib)
            rgb_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            # Deteksi wajah pada gambar
            faces = self.detector(rgb_img)
            if len(faces) == 0:
                return None, None, False

            # Ambil wajah pertama yang terdeteksi
            face = faces[0]

            # Deteksi landmarks wajah
            shape = self.shape_predictor(rgb_img, face)

            # Ekstraksi deskriptor wajah (128-D vector)
            face_descriptor = self.face_rec_model.compute_face_descriptor(rgb_img, shape)

            # Konversi deskriptor ke numpy array
            face_descriptor = np.array(face_descriptor)

            # Ekstraksi koordinat landmarks ke numpy array
            landmarks = np.array([[p.x, p.y] for p in shape.parts()])

            return face_descriptor, landmarks, True

        except Exception as e:
            # Handle error jika ada masalah saat memproses gambar
            print(f"Error processing {image_path}: {str(e)}")
            return None, None, False

def process_dataset(base_path, extractor, subset='train'):
    ##
    # Memproses dataset untuk ekstraksi fitur wajah.
    # Args:
    #    base_path (str): Path dasar ke dataset.
    #    extractor (FaceFeatureExtractor): Objek FaceFeatureExtractor untuk ekstraksi fitur.
    #    subset (str): Subset dataset (misalnya 'augmented' atau 'validation').
    # Returns:
    #    features, landmarks_list, labels, paths: Array fitur, landmarks, label, dan path gambar.

    # List untuk menyimpan hasil ekstraksi
    features = []
    landmarks_list = []
    labels = []
    paths = []

    # Path ke subset dataset
    dataset_path = os.path.join(base_path, subset)

    # Loop melalui setiap folder subjek (kelas)
    for subject_folder in tqdm(sorted(os.listdir(dataset_path)), desc=f'Processing {subset} data'):
        subject_path = os.path.join(dataset_path, subject_folder)
        if not os.path.isdir(subject_path):
            continue

        # Proses setiap gambar dalam folder subjek
        for img_file in sorted(os.listdir(subject_path)):
            if not img_file.endswith('.jpg'):  # Hanya proses file dengan ekstensi .ppm
                continue

            img_path = os.path.join(subject_path, img_file)

            # Ekstraksi fitur dari gambar
            face_descriptor, landmarks, success = extractor.extract_features(img_path)

            if success:
                # Simpan hasil jika berhasil
                features.append(face_descriptor)
                landmarks_list.append(landmarks)
                labels.append(subject_folder)
                paths.append(img_path)

    # Konversi list ke numpy array
    features = np.array(features)
    landmarks_list = np.array(landmarks_list)
    labels = np.array(labels)
    paths = np.array(paths)

    # Simpan hasil ekstraksi ke file npz
    output_file = f'face_features_{subset}.npz'
    np.savez(output_file,
             features=features,
             landmarks=landmarks_list,
             labels=labels,
             paths=paths)

    print(f"\nExtracted features from {len(features)} images")
    print(f"Saved features to {output_file}")

    return features, landmarks_list, labels, paths

### **FUNGSI KLASIFIKASI WAJAH**

In [ ]:

class FaceClassifier:
    def __init__(self):
        self.classifier = SVC(kernel='linear', probability=True)

    def train(self, features, labels):
        """
        Train classifier dengan fitur wajah
        """
        self.classifier.fit(features, labels)

    def evaluate(self, features, true_labels):
        """
        Evaluasi model pada validation set
        """
        predictions = self.classifier.predict(features)
        report = classification_report(true_labels, predictions)
        return predictions, report

    def save_model(self, path='face_classifier.pkl'):
        """
        Simpan model yang sudah ditraining
        """
        with open(path, 'wb') as f:
            pickle.dump(self.classifier, f)

    @classmethod
    def load_model(cls, path='face_classifier.pkl'):
        """
        Load model yang sudah disimpan
        """
        classifier = cls()
        with open(path, 'rb') as f:
            classifier.classifier = pickle.load(f)
        return classifier

def main():
    # Inisialisasi feature extractor
    extractor = FaceFeatureExtractor()

    # Path ke dataset
    dataset_base_path = './dataset'

    # Proses training data
    print("Processing training data...")
    train_features, train_landmarks, train_labels, train_paths = process_dataset(
        dataset_base_path, extractor, subset='train'
    )

    # Proses validation data
    print("\nProcessing validation data...")
    val_features, val_landmarks, val_labels, val_paths = process_dataset(
        dataset_base_path, extractor, subset='val'
    )

    # Train classifier
    print("\nTraining classifier...")
    classifier = FaceClassifier()
    classifier.train(train_features, train_labels)

    # Evaluate pada validation set
    print("\nEvaluating on validation set...")
    predictions, report = classifier.evaluate(val_features, val_labels)
    print("\nClassification Report:")
    print(report)

    # Simpan model
    classifier.save_model('./model/face_classifier.pkl')
    print("\nModel saved to face_classifier.pkl")

if __name__ == "__main__":
    main()

### **KLASIFIKASI SVM DAN ANALISIS HASIL TIAP KELAS**

In [ ]:
import numpy as np
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

def load_features(train_npz_path, val_npz_path):
    """
    Load fitur dari file NPZ
    """
    # Memuat data training
    train_data = np.load(train_npz_path)
    X_train = train_data['features']
    y_train = train_data['labels']

    # Memuat data validasi
    val_data = np.load(val_npz_path)
    X_val = val_data['features']
    y_val = val_data['labels']

    return X_train, y_train, X_val, y_val

def train_svm_classifier(X_train, y_train, X_val, y_val):
    """
    Train SVM classifier dan evaluasi performanya
    """
    # Inisialisasi SVM classifier
    svm = LinearSVC(random_state=42, max_iter=5000)

    # Melatih SVM classifier
    print("Training SVM classifier...")
    svm.fit(X_train, y_train)

    # Prediksi dari hasil training
    train_predictions = svm.predict(X_train)
    val_predictions = svm.predict(X_val)

    return train_predictions, y_train, val_predictions, y_val

# Plot confusion matrix untuk beberapa kelas pertama
def plot_confusion_matrix(y_true, y_pred, title, num_classes_to_show=10):

    # Mencari label yang unik
    unique_labels = sorted(list(set(y_true)))[:num_classes_to_show]

    # Filter data untuk kelas yang akan ditampilkan
    mask_true = np.isin(y_true, unique_labels)
    mask_pred = np.isin(y_pred, unique_labels)
    mask = mask_true & mask_pred

    y_true_filtered = [y for i, y in enumerate(y_true) if mask[i]]
    y_pred_filtered = [y for i, y in enumerate(y_pred) if mask[i]]

    # Buat confusion matrix
    cm = confusion_matrix(y_true_filtered, y_pred_filtered, labels=unique_labels)

    # Plot
    plt.figure(figsize=(12, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=unique_labels,
                yticklabels=unique_labels)
    plt.title(f'{title}\n(Showing first {num_classes_to_show} classes)')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()

def calculate_per_class_accuracy(y_true, y_pred):
   # Hitung akurasi per kelas
    unique_labels = sorted(list(set(y_true)))
    per_class_acc = {}

    for label in unique_labels:
        mask = (y_true == label)
        correct = (y_true[mask] == y_pred[mask]).sum()
        total = mask.sum()
        per_class_acc[label] = (correct / total) * 100

    return per_class_acc

def display_results(train_pred, train_true, val_pred, val_true):
    # Tampilkan hasil evaluasi model

    # Menhitung akurasi keseluruhan
    train_acc = accuracy_score(train_true, train_pred)
    val_acc = accuracy_score(val_true, val_pred)

    print("\n=== Overall Model Performance ===")
    print(f"Training Accuracy: {train_acc*100:.2f}%")
    print(f"Validation Accuracy: {val_acc*100:.2f}%")

    # Menghitung akurasi tiap kelas
    print("\n=== Per-Class Accuracies ===")
    train_class_acc = calculate_per_class_accuracy(train_true, train_pred)
    val_class_acc = calculate_per_class_accuracy(val_true, val_pred)

    print("\nTop 5 Best Performing Classes (Training):")
    for label, acc in sorted(train_class_acc.items(), key=lambda x: x[1], reverse=True)[:5]:
        print(f"Class {label}: {acc:.2f}%")

    print("\nTop 5 Best Performing Classes (Validation):")
    for label, acc in sorted(val_class_acc.items(), key=lambda x: x[1], reverse=True)[:5]:
        print(f"Class {label}: {acc:.2f}%")

    # Report hasil klasifikasi
    print("\n=== Training Set Classification Report ===")
    print(classification_report(train_true, train_pred))

    print("\n=== Validation Set Classification Report ===")
    print(classification_report(val_true, val_pred))

    # Plot confusion matrix 10 kelas pertama
    print("\nPlotting confusion matrices...")
    plot_confusion_matrix(train_true, train_pred, 'Training Set Confusion Matrix')
    plot_confusion_matrix(val_true, val_pred, 'Validation Set Confusion Matrix')

def main():
    # Path ke file NPZ
    train_npz_path = './face_features_train.npz'
    val_npz_path = './face_features_val.npz'

    try:
        # Load Fitur
        print("Loading features from NPZ files...")
        X_train, y_train, X_val, y_val = load_features(train_npz_path, val_npz_path)

        print(f"Training samples: {X_train.shape}")
        print(f"Validation samples: {X_val.shape}")

        # === ADD THIS BLOCK ===

        def plot_learning_curve(estimator, X, y, title="Learning Curve"):
            """
            Plot learning curve showing training and validation accuracy vs. training set size.
            """
            train_sizes, train_scores, val_scores = learning_curve(
                estimator, X, y,
                cv=3,  # 3-fold cross-validation
                n_jobs=-1,
                train_sizes=np.linspace(0.1, 1.0, 10),
                random_state=42
            )

            train_mean = np.mean(train_scores, axis=1)
            train_std = np.std(train_scores, axis=1)
            val_mean = np.mean(val_scores, axis=1)
            val_std = np.std(val_scores, axis=1)

            plt.figure(figsize=(10, 6))
            plt.plot(train_sizes, train_mean, 'o-', color='blue', label='Training Accuracy')
            plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color='blue')

            plt.plot(train_sizes, val_mean, 'o-', color='red', label='Validation Accuracy')
            plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.1, color='red')

            plt.title(title)
            plt.xlabel('Training Set Size')
            plt.ylabel('Accuracy')
            plt.legend()
            plt.grid(True)
            plt.tight_layout()
            plt.show()

        # Create an SVM instance for learning curve (same as used in training)
        svm_for_curve = LinearSVC(random_state=42, max_iter=5000)

        print("\nGenerating learning curve...")
        plot_learning_curve(svm_for_curve, X_train, y_train, title="SVM Learning Curve (Accuracy vs. Training Size)")


        # Train dan evaluasi model
        train_pred, train_true, val_pred, val_true = train_svm_classifier(
            X_train, y_train, X_val, y_val
        )

        # Tampilkan hasil
        display_results(train_pred, train_true, val_pred, val_true)

    except Exception as e:
        print(f"Error occurred: {str(e)}")
        raise

if __name__ == "__main__":
    main()

### **ANALISIS AKURASI TIAP _CLASS_**

In [ ]:
import numpy as np
from sklearn.svm import LinearSVC
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.preprocessing import LabelEncoder
from collections import Counter

def analyze_per_class(true_labels, predictions, le, dataset_name=""):

    # Menganalisis hasil klasifikasi untuk setiap kelas
    print(f"\n=== ANALISIS PER KELAS - {dataset_name} ===")
    print("Format: Kelas | Total Data | Berhasil | Gagal | Akurasi")
    print("-" * 60)

    # Konversi prediksi dan true labels ke label asli
    true_labels_original = le.inverse_transform(true_labels)
    pred_labels_original = le.inverse_transform(predictions)

    # Hitung statistik untuk setiap kelas
    class_stats = {}

    # Hitung total data per kelas
    class_counts = Counter(true_labels_original)

    for class_name in sorted(class_counts.keys()):
        # Ambil indeks untuk kelas ini
        class_indices = np.where(true_labels_original == class_name)[0]

        # Hitung total data
        total = len(class_indices)

        # Hitung yang benar
        correct = sum(true_labels_original[i] == pred_labels_original[i]
                     for i in class_indices)

        # Hitung yang salah
        wrong = total - correct

        # Hitung akurasi
        accuracy = (correct / total) * 100

        # Simpan statistik
        class_stats[class_name] = {
            'total': total,
            'correct': correct,
            'wrong': wrong,
            'accuracy': accuracy
        }

        # Tampilkan hasil
        print(f"{class_name:6} | {total:10d} | {correct:8d} | {wrong:5d} | {accuracy:6.2f}%")

    return class_stats

def analyze_svm_classifications(train_npz_path, val_npz_path):
    # Load data train dan validation
    train_data = np.load(train_npz_path)
    val_data = np.load(val_npz_path)

    # Ekstrak features dan labels
    X_train = train_data['features']
    y_train = train_data['labels']
    X_val = val_data['features']
    y_val = val_data['labels']

    # Encoding label
    le = LabelEncoder()
    # Gabungkan semua label untuk memastikan encoder mengenal semua kelas
    all_labels = np.concatenate([y_train, y_val])
    le.fit(all_labels)

    # Transform label
    y_train_encoded = le.transform(y_train)
    y_val_encoded = le.transform(y_val)

    # Inisialisasi dan latih model SVM
    svm_model = LinearSVC(random_state=42, max_iter=5000)
    svm_model.fit(X_train, y_train_encoded)

    # Prediksi
    train_predictions = svm_model.predict(X_train)
    val_predictions = svm_model.predict(X_val)

    # Analisis hasil keseluruhan training
    train_correct = np.sum(train_predictions == y_train_encoded)
    train_total = len(y_train_encoded)
    train_wrong = train_total - train_correct
    train_accuracy = (train_correct / train_total) * 100

    print("=== HASIL KLASIFIKASI KESELURUHAN - TRAINING ===")
    print(f"Jumlah total data training: {train_total}")
    print(f"Jumlah klasifikasi berhasil: {train_correct}")
    print(f"Jumlah klasifikasi gagal: {train_wrong}")
    print(f"Akurasi training: {train_accuracy:.2f}%")

    # Analisis per kelas untuk data training
    train_class_stats = analyze_per_class(y_train_encoded, train_predictions, le, "TRAINING")

    # Analisis hasil keseluruhan validation
    val_correct = np.sum(val_predictions == y_val_encoded)
    val_total = len(y_val_encoded)
    val_wrong = val_total - val_correct
    val_accuracy = (val_correct / val_total) * 100

    print("\n=== HASIL KLASIFIKASI KESELURUHAN - VALIDATION ===")
    print(f"Jumlah total data validation: {val_total}")
    print(f"Jumlah klasifikasi berhasil: {val_correct}")
    print(f"Jumlah klasifikasi gagal: {val_wrong}")
    print(f"Akurasi validation: {val_accuracy:.2f}%")

    # Analisis per kelas untuk data validation
    val_class_stats = analyze_per_class(y_val_encoded, val_predictions, le, "VALIDATION")

    # Tampilkan kelas dengan akurasi terendah (5 terendah)
    print("\n=== 5 KELAS DENGAN AKURASI TERENDAH - VALIDATION ===")
    worst_classes = sorted(val_class_stats.items(),
                         key=lambda x: x[1]['accuracy'])[:5]
    for class_name, stats in worst_classes:
        print(f"Kelas {class_name}: {stats['accuracy']:.2f}% "
              f"(Benar: {stats['correct']}/{stats['total']})")

    return {
        'train_results': {
            'total': train_total,
            'correct': train_correct,
            'wrong': train_wrong,
            'accuracy': train_accuracy,
            'per_class_stats': train_class_stats
        },
        'val_results': {
            'total': val_total,
            'correct': val_correct,
            'wrong': val_wrong,
            'accuracy': val_accuracy,
            'per_class_stats': val_class_stats
        }
    }

if __name__ == "__main__":
    train_npz_path = "./face_features_train.npz"
    val_npz_path = "./face_features_val.npz"
    results = analyze_svm_classifications(train_npz_path, val_npz_path)
